# Ablation Cost Attribution

Component contributions to baseline-optimized route costs.

In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import numpy as np
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [ ]:
gdf = gpd.read_parquet("../results/best_elites_per_test_case.geoparquet")

# Filter to baseline-optimized routes only
gdf_baseline = gdf[gdf.forcing_scenario_name == "baseline"].copy()
gdf_baseline

In [ ]:
# Cost increases when components removed (%)
gdf_baseline = gdf_baseline.assign(
    delta_currents=(gdf_baseline.ablation_cost_no_currents - gdf_baseline.ablation_cost_baseline)
                   / gdf_baseline.ablation_cost_baseline * 100,
    delta_waves=(gdf_baseline.ablation_cost_no_waves - gdf_baseline.ablation_cost_baseline)
                / gdf_baseline.ablation_cost_baseline * 100,
    delta_winds=(gdf_baseline.ablation_cost_no_winds - gdf_baseline.ablation_cost_baseline)
                / gdf_baseline.ablation_cost_baseline * 100,
    journey_month=gdf_baseline.journey_time_start.str[:7],
)
gdf_baseline

In [ ]:
# Pivot to long form for seaborn
df_long = gdf_baseline.melt(
    id_vars=["journey_month", "journey_name", "journey_speed_knots"],
    value_vars=["delta_currents", "delta_waves", "delta_winds"],
    var_name="component",
    value_name="cost_increase_pct",
)
df_long

In [ ]:
# Use seaborn FacetGrid for automatic styling
g = sns.FacetGrid(
    df_long,
    col="journey_speed_knots",
    hue="component",
    height=4,
    aspect=1.2,
)
g.map_dataframe(sns.lineplot, x="journey_month", y="cost_increase_pct", style="journey_name")
g.set_titles("{col_name} knots")
g.set_axis_labels("Month", "Cost increase (%)")
g.add_legend()

g.savefig("../figures/022_ablation_cost_attribution.pdf", dpi=200)
g.savefig("../figures/022_ablation_cost_attribution.png", dpi=200)

In [ ]:
# Aggregate at 10 knots using pandas groupby
summary = (
    gdf_baseline[gdf_baseline.journey_speed_knots == 10.0]
    [["delta_currents", "delta_waves", "delta_winds"]]
    .agg(["mean", "std", "min", "max"])
    .T
)
summary.columns = ["Mean (%)", "Std Dev", "Min", "Max"]
summary.round(2)

In [ ]:
summary.to_csv("../results/ablation_cost_summary.csv")